# 15.7 Router 怎麼收到學習訊號？


router選中了某位expert，怎麼知道下次要增加或減少對它的偏好？前置是[top-k選擇](https://birdhackor.github.io/tiny-perceptron-vlm/15.4.html)與[合併比例](https://birdhackor.github.io/tiny-perceptron-vlm/15.5.html)，以及[梯度](https://birdhackor.github.io/tiny-perceptron-vlm/first-steps.html#W.6)：梯度告訴我們，微小改動某個可學數字會讓誤差往哪邊變。本節只追蹤「保留的比例」這條路，因為選中的整數索引不能直接像連續數字一樣微分。

假設兩位的router分數是 `[log(2),0]`，softmax比例為 `[2/3,1/3]`，top-1選第一位。這位的輸出固定為2，希望最後輸出接近0，於是用輸出的平方作誤差。若保留原比例，輸出是 `2×2/3=4/3`，誤差16/9；微調第一位的分數會改變2/3這個比例，因此誤差能沿比例回傳給router。

但若top-1也把唯一比例除以選中總和，結果成為 `p/p=1`，不論p微小變大或變小，輸出都是2，誤差都是4。這條路給router的梯度便為零。不是PyTorch忘記學習，而是我們把唯一可以連續改變的係數消掉了。


In [ ]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [ ]:
import math
import torch

for renormalize in (False, True):
    scores = torch.tensor([math.log(2), 0.0], requires_grad=True)
    prob = scores.softmax(-1)
    p = prob.topk(1).values[0]
    gate = p / p if renormalize else p
    output = gate * 2
    loss = output.square()
    loss.backward()
    print("重新正規化", renormalize, "輸出", round(output.item(), 4), "梯度", scores.grad.round(decimals=4).tolist())

兩輪各建立一份scores，避免前一輪的梯度累加干擾。`requires_grad=True`讓PyTorch記錄從scores到loss的運算，`backward`沿這些運算求導數。未重新正規化時輸出1.3333，梯度約 `[1.1852,-1.1852]`；若沿梯度的反方向更新，第一個分數減小、第二個增加，選中者的比例降低，符合我們想降低輸出的目標。重新正規化時輸出2.0，梯度兩項都是0。

這個數值例子不代表所有MoE都必須使用相同約定。其他方法可能設計不同的連續近似，或加入額外的路由學習目標；本教材的top-1保留原softmax機率，讓任務誤差能透過gate教router。top-2重新正規化後仍有兩個相對比例可變，不會自動變成恆定係數，所以不能把top-1的結論直接套到top-2。

練習只把expert輸出2改成1。先預測未正規化的輸出減半，而平方誤差對scores的梯度變成原來四分之一，約 `[0.2963,-0.2963]`；重跑核對。重新正規化那條路仍是零梯度。這能分清「誤差大小變了」與「router是否保有學習路徑」兩個問題。
